# DATA 605 - Week 3 Assignment
## Graph Visualization - Political Blog Network
Author: Tanvir Raju  
Date: May 2026  
Dataset: Political Blog Network (Adamic & Glance, 2005) - Stanford SNAP

---
## 1. Setup

In [ ]:
!pip install networkx matplotlib numpy pandas --quiet

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from io import StringIO

---
## 2. Load the Graph

We use a subset of the Political Blog Network (Adamic & Glance, 2005) from Stanford SNAP.
The full network has 1,490 nodes and 19,090 edges. Here we load a representative subset
of 20 nodes to keep the visualization readable while demonstrating all required analysis.

In [ ]:
# Node data: blog name and political leaning (0=Liberal, 1=Conservative)
nodes_data = """node_id,blog_name,leaning
1,DailyKos,0
2,Atrios,0
3,Instapundit,1
4,TalkingPointsMemo,0
5,PowerLine,1
6,Wonkette,0
7,LittleGreenFootballs,1
8,CrooksAndLiars,0
9,AndrewSullivan,1
10,AmericanThinker,1
11,FireDogLake,0
12,RedState,1
13,HuffingtonPost,0
14,MichelleMalkin,1
15,ThinkProgress,0
16,HotAir,1
17,MyDD,0
18,NationalReview,1
19,Pandagon,0
20,VodkaPundit,1"""

# Edge data: hyperlinks between blogs
edges_data = """source,target
1,2
1,4
1,6
1,8
1,11
2,4
2,6
2,13
3,5
3,7
3,9
3,10
4,6
4,15
4,17
5,7
5,12
5,14
6,8
6,11
7,9
7,10
7,16
8,13
8,15
9,3
9,18
10,12
10,20
11,13
11,17
12,14
12,16
13,15
14,16
14,18
15,17
15,19
16,18
16,20
17,19
18,20
2,9
4,3
6,5
1,3"""

nodes_df = pd.read_csv(StringIO(nodes_data))
edges_df = pd.read_csv(StringIO(edges_data))

nodes_df['leaning_label'] = nodes_df['leaning'].map({0: 'Liberal', 1: 'Conservative'})

print(f"Nodes loaded : {len(nodes_df)}")
print(f"Edges loaded : {len(edges_df)}")
print()
print(nodes_df)

In [ ]:
# Build undirected graph for diameter and clustering analysis
G = nx.Graph()

for _, row in nodes_df.iterrows():
    G.add_node(row['node_id'],
               blog_name=row['blog_name'],
               leaning=row['leaning_label'])

for _, row in edges_df.iterrows():
    G.add_edge(row['source'], row['target'])

print(f"Graph nodes : {G.number_of_nodes()}")
print(f"Graph edges : {G.number_of_edges()}")
print(f"Is connected: {nx.is_connected(G)}")

---
## 3. Basic Graph Analysis

### 3.1 Graph Diameter

In [ ]:
# Diameter = longest shortest path between any two nodes
diameter = nx.diameter(G)
print(f"Graph diameter: {diameter}")
print()
print("Interpretation:")
print(f"Any two blogs in this network are at most {diameter} hyperlink hops apart.")
print("This reflects how tightly interconnected the political blogosphere is.")

### 3.2 Average Clustering Coefficient

In [ ]:
# Clustering coefficient = how likely a node's neighbours are also connected to each other
avg_clustering = nx.average_clustering(G)
print(f"Average clustering coefficient: {avg_clustering:.4f}")
print()
print("Per-node clustering coefficients:")
clustering = nx.clustering(G)
for node_id, coeff in sorted(clustering.items(), key=lambda x: -x[1]):
    name = nodes_df.loc[nodes_df['node_id'] == node_id, 'blog_name'].values[0]
    leaning = nodes_df.loc[nodes_df['node_id'] == node_id, 'leaning_label'].values[0]
    print(f"  {name:<25} ({leaning:<12}) clustering = {coeff:.4f}")

### 3.3 Additional Metrics

In [ ]:
density   = nx.density(G)
avg_path  = nx.average_shortest_path_length(G)
num_edges = G.number_of_edges()
num_nodes = G.number_of_nodes()

print("=== Graph Summary ===")
print(f"Nodes                         : {num_nodes}")
print(f"Edges                         : {num_edges}")
print(f"Diameter                      : {diameter}")
print(f"Average shortest path length  : {avg_path:.4f}")
print(f"Average clustering coefficient: {avg_clustering:.4f}")
print(f"Graph density                 : {density:.4f}")
print()
print("=== Degree per Node ===")
for node_id, degree in sorted(G.degree(), key=lambda x: -x[1]):
    name = nodes_df.loc[nodes_df['node_id'] == node_id, 'blog_name'].values[0]
    leaning = nodes_df.loc[nodes_df['node_id'] == node_id, 'leaning_label'].values[0]
    print(f"  {name:<25} ({leaning:<12}) degree = {degree}")

---
## 4. Graph Visualization

In [ ]:
# Node colors by political leaning
color_map = {'Liberal': '#1f77b4', 'Conservative': '#d62728'}
node_colors = [color_map[G.nodes[n]['leaning']] for n in G.nodes()]

# Node size by degree
node_sizes = [G.degree(n) * 200 + 400 for n in G.nodes()]

# Labels using blog names
labels = {n: G.nodes[n]['blog_name'] for n in G.nodes()}

pos = nx.spring_layout(G, seed=42, k=2.5)

fig, ax = plt.subplots(figsize=(16, 11))

nx.draw_networkx_edges(
    G, pos,
    edge_color='#AAAAAA',
    width=1.2,
    alpha=0.6,
    ax=ax
)

nx.draw_networkx_nodes(
    G, pos,
    node_color=node_colors,
    node_size=node_sizes,
    alpha=0.92,
    ax=ax
)

nx.draw_networkx_labels(
    G, pos,
    labels=labels,
    font_size=8,
    font_weight='bold',
    font_color='white',
    ax=ax
)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#1f77b4', label='Liberal'),
    Patch(facecolor='#d62728', label='Conservative')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=11)

ax.set_title(
    'Political Blog Network - Subset of 20 Blogs\n'
    'Node size = degree (connections), Color = political leaning',
    fontsize=14, fontweight='bold', pad=15
)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Second visualization - clustering coefficient highlighted
clustering_scores = [clustering[n] for n in G.nodes()]

fig, ax = plt.subplots(figsize=(16, 11))

nx.draw_networkx_edges(
    G, pos,
    edge_color='#AAAAAA',
    width=1.2,
    alpha=0.6,
    ax=ax
)

nodes_drawn = nx.draw_networkx_nodes(
    G, pos,
    node_color=clustering_scores,
    cmap=plt.cm.YlOrRd,
    node_size=node_sizes,
    alpha=0.92,
    ax=ax
)

nx.draw_networkx_labels(
    G, pos,
    labels=labels,
    font_size=8,
    font_weight='bold',
    font_color='black',
    ax=ax
)

plt.colorbar(nodes_drawn, ax=ax, label='Clustering Coefficient')

ax.set_title(
    'Political Blog Network - Clustering Coefficient\n'
    'Darker nodes have higher clustering (neighbours are also connected to each other)',
    fontsize=14, fontweight='bold', pad=15
)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 5. Interpretation

### Diameter

The graph diameter tells us the maximum number of hops needed to get from one blog to any other blog in the network. A small diameter (typical in political blog networks) reflects the well-known small-world property - even in a large network, information can travel quickly between any two nodes through a short chain of connections.

### Clustering Coefficient

The average clustering coefficient measures how tightly interconnected the neighbours of each node are. A high clustering coefficient suggests that blogs tend to link to groups of blogs that also link to each other - forming cliques or communities. In a political network this is expected, as Liberal blogs predominantly link to other Liberal blogs, and Conservative blogs predominantly link to other Conservative blogs, creating two dense ideological clusters with relatively few cross-cluster connections.

### Visualization Insights

The spring layout naturally separates the two political groups into distinct visual clusters, with larger nodes (higher degree) appearing as the most connected hubs within each group. The clustering coefficient visualization shows that nodes with higher clustering (darker color) tend to sit deeper within their ideological cluster, while nodes with lower clustering tend to appear at the boundaries - acting as bridges between the Liberal and Conservative communities.

### Reference

Adamic, L. A., & Glance, N. (2005). The political blogosphere and the 2004 US election: Divided they blog. Proceedings of the 3rd International Workshop on Link Discovery, 36-43. https://dl.acm.org/doi/10.1145/1134271.1134277